# Momentum One — GPU Edition (Bot 1.5)
### Train thousands of markets at once — and never lose progress

**FIRST: turn on the GPU.** Menu -> **Runtime** -> **Change runtime type** -> **L4 GPU** (or **A100** for ~2× more) -> **Save**.

Then run the cells top to bottom.


## Cell 1 — load the bot + connect your Drive (run once)
A window pops up to connect your **Google Drive** (click Allow) — that's how the best brain survives Colab shutting off.


In [ ]:
import torch, os, shutil
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NONE  <-  Runtime > Change runtime type > L4 GPU, then re-run')
from google.colab import drive
drive.mount('/content/drive')
!git clone -q https://github.com/monty313/the-truth.git
%cd the-truth
!pip -q install pyyaml >/dev/null 2>&1
DR = '/content/drive/MyDrive/momentum_gpu'
os.makedirs(DR, exist_ok=True)
for f in os.listdir('artifacts/checkpoints'):
    if f.endswith('.pt') and not os.path.exists(f'{DR}/{f}'):
        shutil.copy(f'artifacts/checkpoints/{f}', f'{DR}/{f}')
shutil.rmtree('artifacts/checkpoints', ignore_errors=True)
os.symlink(DR, 'artifacts/checkpoints')
print('best brain saved to your Drive:', DR); print('ready.')


## Cell 2 — START TRAINING
Now with your settings baked in: **60% of practice pinned to 3% target / 3.5% risk** (the reachable band),
40% random for range, **decisions every 5 minutes**, record checked **once an hour** — the fast setup.

Runs until Colab stops it. **Re-run to continue** from your best. Out of memory? change `--env-mb 384` to `256`.

Want to change the mix? Use the **Training Configurator** to build a custom command.


In [ ]:
!python scripts/gpu_train.py --instances 8000 --minutes 1440 --patience 100000 \
  --focus-frac 0.6 --focus-target 3 --focus-risk 3.5 \
  --decide-every 5 --eval-every 30 --epochs 1 --K 24 --env-mb 384


## Cell 3 — how is it doing?
Best streak so far, and every saved record. `pass0012` = 12 cleared days in a row.


In [ ]:
!cat artifacts/checkpoints/gpu_progress.json
print('\n--- saved records ---')
!ls -1 artifacts/checkpoints/ 2>/dev/null | grep gpu_pass || echo 'no records yet'


## Cell 4 — the good brain (already on your Drive)
Best brain auto-saves to **MyDrive / momentum_gpu / gpu_best.pt**. This cell also copies it to your computer.


In [ ]:
from google.colab import files
files.download('artifacts/checkpoints/gpu_best.pt')


## Cell 5 — double-check on the REAL sim (optional, ~5 min)
Runs the saved brain on your exact simulator (not the fast copy) so you can trust it.


In [ ]:
!python scripts/gpu_validate.py --csv data/XAUUSD_curriculum_2026.csv --ckpt gpu_best
